In [ ]:
# ==== tabular4_feature_alignment (single-site, global stratified split) ====
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

BASE = Path(".")  
SRC  = BASE / "diabetesdata.csv"  

FEATURES = [
    'Age','Sex','BMI','GenHlth','HighBP','DiffWalk','HighChol','HeartDiseaseorAttack'
]
LABEL = "label"

SPLIT_RATIO = (0.6, 0.2, 0.2)  # train / val / test



In [15]:
# 读取并对齐列：把 Diabetes_012 改成 label，并选 8 个特征
df = pd.read_csv(SRC)
df = df[['Diabetes_012'] + FEATURES].rename(columns={'Diabetes_012': LABEL}).copy()

# 轻量数值转换（保留 NaN，后面要用来生成 *_mask）
for c in [LABEL] + FEATURES:
    df[c] = pd.to_numeric(df[c], errors='coerce')

print("Aligned base shape:", df.shape)
display(df.head())



Aligned base shape: (253680, 9)


,label,Age,Sex,BMI,GenHlth,HighBP,DiffWalk,HighChol,HeartDiseaseorAttack
0,0,9.0,0.0,40.0,5.0,1.0,1.0,1.0,NaN
1,0,7.0,0.0,25.0,3.0,0.0,0.0,0.0,0.0
2,0,9.0,0.0,28.0,5.0,1.0,1.0,1.0,NaN
3,0,11.0,0.0,27.0,2.0,1.0,0.0,0.0,0.0
4,0,11.0,0.0,24.0,2.0,1.0,0.0,1.0,0.0


In [16]:
aligned_path = BASE / "tabular_rex_aligned_clean.csv"
df.to_csv(aligned_path, index=False)
print("[OK] saved aligned:", aligned_path, df.shape)


[OK] saved aligned: tabular_rex_aligned_clean.csv (253680, 9)


In [ ]:
# === Z-score stats for tabular_rex -> JSON (single cell) ===
import json, math
from pathlib import Path
import pandas as pd

# 1) 路径与特征（按你仓库习惯可改）
site_name = "tabular_rex"
in_csv    = Path("../tabular_rex/tabular_rex_aligned_clean.csv")
out_json  = Path("../tabular_rex/tabular_rex_stats.json")

TEMPLATE_FEATURES = [
    "Age", "Sex", "BMI", "GenHlth",
    "HighBP", "DiffWalk", "HighChol", "HeartDiseaseorAttack",
]

# 2) 读取
df = pd.read_csv(in_csv)

# 3) 计算均值/方差（用于 z-score： (x-mean)/std ，std=sqrt(var) ）
feature_means, feature_vars = {}, {}
missing, all_nan, constant = [], [], []

for feat in TEMPLATE_FEATURES:
    if feat not in df.columns:
        missing.append(feat)
        continue

    col = pd.to_numeric(df[feat], errors="coerce")  # 非数值转为 NaN
    if col.notna().sum() == 0:
        # 全是 NaN：不参与归一化，标记出来（用 None 占位更明确）
        all_nan.append(feat)
        feature_means[feat] = None
        feature_vars[feat]  = None
        continue

    mu   = float(col.mean(skipna=True))
    # 用总体方差（ddof=0），训练里更常见；若全常数则置 var=0 并记录
    var  = float(col.var(skipna=True, ddof=0))
    if not math.isfinite(var):
        var = 0.0
    if var == 0.0:
        constant.append(feat)

    feature_means[feat] = mu
    feature_vars[feat]  = var

stats = {
    "site_name": site_name,
    "n_samples": int(df.shape[0]),
    "feature_means": feature_means,
    "feature_vars": feature_vars,
    "zscore_note": "Use z = (x - mean) / std ; std = sqrt(var) with ddof=0. Constant columns have var=0.",
    "missing_features": missing,
    "all_nan_features": all_nan,
    "constant_features": constant,
}

out_json.parent.mkdir(parents=True, exist_ok=True)
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)
print(f"[OK] Saved -> {out_json}")

# 额外提示信息（可检查为何你之前某些列是 0）
print(f"Missing: {missing}")
print(f"All-NaN: {all_nan}")
print(f"Constant(var=0): {constant}")


FileNotFoundError: [Errno 2] No such file or directory: '../tabular_rex/tabular_rex_aligned_mask_templated.csv'

In [17]:
# 第一次切：train vs temp(val+test)
train_size = SPLIT_RATIO[0]  # 0.6
df_train, df_temp = train_test_split(
    df, test_size=1-train_size, stratify=df[LABEL], random_state=42
)
# 第二次切：temp → val / test （对半 0.5）
val_ratio = SPLIT_RATIO[1] / (SPLIT_RATIO[1] + SPLIT_RATIO[2])  # 0.2 / 0.4 = 0.5
df_val, df_test = train_test_split(
    df_temp, test_size=1-val_ratio, stratify=df_temp[LABEL], random_state=42
)

print("shapes:", df_train.shape, df_val.shape, df_test.shape)



shapes: (152208, 9) (50736, 9) (50736, 9)


In [18]:
train_out = BASE / "tabular_rex_train.csv"
val_out   = BASE / "tabular_rex_val.csv"
test_out  = BASE / "tabular_rex_test.csv"

df_train.to_csv(train_out, index=False)
df_val.to_csv(val_out, index=False)
df_test.to_csv(test_out, index=False)
print("[OK] wrote:", train_out, val_out, test_out)



[OK] wrote: tabular_rex_train.csv tabular_rex_val.csv tabular_rex_test.csv


In [19]:
FILES = [train_out, val_out, test_out]

for f in FILES:
    d = pd.read_csv(f)
    present = [c for c in FEATURES if c in d.columns]
    for col in present:
        mcol = f"{col}_mask"
        if mcol not in d.columns:
            d[mcol] = (~pd.isna(d[col])).astype("int8")
    d.to_csv(f, index=False)
    print(f"[OK] add masks -> {f} (+{sum(f'{c}_mask' in d.columns for c in present)} cols)")


[OK] add masks -> tabular_rex_train.csv (+8 cols)
[OK] add masks -> tabular_rex_val.csv (+8 cols)
[OK] add masks -> tabular_rex_test.csv (+8 cols)


In [20]:
def ratio(x): return x[LABEL].value_counts(normalize=True).round(3).to_dict()

al = pd.read_csv("tabular_rex_aligned_clean.csv")
tr = pd.read_csv("tabular_rex_train.csv")
va = pd.read_csv("tabular_rex_val.csv")
te = pd.read_csv("tabular_rex_test.csv")

print("rows:", len(al), len(tr), len(va), len(te), "sum=", len(tr)+len(va)+len(te))
print("label ratio | aligned:", ratio(al), "| train:", ratio(tr), "| val:", ratio(va), "| test:", ratio(te))


rows: 253680 152208 50736 50736 sum= 253680
label ratio | aligned: {0: 0.842, 2: 0.139, 1: 0.018} | train: {0: 0.842, 2: 0.139, 1: 0.018} | val: {0: 0.842, 2: 0.139, 1: 0.018} | test: {0: 0.842, 2: 0.139, 1: 0.018}
